In [1]:
import pandas as pd
from sklearn.model_selection import StratifiedKFold

#PREPROCESSING:

####Read in raw training set:

In [2]:
DF = pd.read_csv('../data/raw/train.csv')
print(DF.head())

       id             model_a              model_b  \
0   30192  gpt-4-1106-preview           gpt-4-0613   
1   53567           koala-13b           gpt-4-0613   
2   65089  gpt-3.5-turbo-0613       mistral-medium   
3   96401    llama-2-13b-chat  mistral-7b-instruct   
4  198779           koala-13b   gpt-3.5-turbo-0314   

                                              prompt  \
0  ["Is it morally right to try to have a certain...   
1  ["What is the difference between marriage lice...   
2  ["explain function calling. how would you call...   
3  ["How can I create a test set for a very rare ...   
4  ["What is the best way to travel from Tel-Aviv...   

                                          response_a  \
0  ["The question of whether it is morally right ...   
1  ["A marriage license is a legal document that ...   
2  ["Function calling is the process of invoking ...   
3  ["Creating a test set for a very rare category...   
4  ["The best way to travel from Tel Aviv to Jeru...   

 

####Clean & reformat dataset to just include required columns

In [3]:
longDF = pd.concat( [
    DF[['model_a', 'prompt', 'response_a']].rename(columns={'model_a': 'LLM_name', 'prompt': 'LLM_Input', 'response_a': 'LLM_output'}),
    DF[['model_b', 'prompt', 'response_b']].rename(columns={'model_b': 'LLM_name','prompt': 'LLM_Input', 'response_b': 'LLM_output'}),
], ignore_index=True)

print(longDF.head())

DF = longDF

             LLM_name                                          LLM_Input  \
0  gpt-4-1106-preview  ["Is it morally right to try to have a certain...   
1           koala-13b  ["What is the difference between marriage lice...   
2  gpt-3.5-turbo-0613  ["explain function calling. how would you call...   
3    llama-2-13b-chat  ["How can I create a test set for a very rare ...   
4           koala-13b  ["What is the best way to travel from Tel-Aviv...   

                                          LLM_output  
0  ["The question of whether it is morally right ...  
1  ["A marriage license is a legal document that ...  
2  ["Function calling is the process of invoking ...  
3  ["Creating a test set for a very rare category...  
4  ["The best way to travel from Tel Aviv to Jeru...  


####Get counts of each different model

In [4]:
print(DF["LLM_name"].value_counts().to_string())

LLM_name
gpt-4-1106-preview                7387
gpt-3.5-turbo-0613                7083
gpt-4-0613                        6165
claude-2.1                        5583
claude-instant-1                  4136
gpt-4-0314                        4122
claude-1                          3978
vicuna-33b                        3720
mixtral-8x7b-instruct-v0.1        3545
vicuna-13b                        3448
llama-2-70b-chat                  3428
gpt-3.5-turbo-1106                3352
mistral-medium                    3315
llama-2-13b-chat                  2607
claude-2.0                        2456
zephyr-7b-beta                    2401
palm-2                            1977
llama-2-7b-chat                   1793
wizardlm-70b                      1644
openchat-3.5                      1632
mistral-7b-instruct               1617
koala-13b                         1598
vicuna-7b                         1591
wizardlm-13b                      1580
oasst-pythia-12b                  1494
gemini-pro-dev-a

####Narrowed down to just 3 common LLMs: gpt-4-0613, claude-2.1, llama-2-70b-chat

In [5]:
DF = DF.loc[DF['LLM_name'].isin(['gpt-4-0613', 'claude-2.1', 'llama-2-70b-chat'])]
DF.head()

,LLM_name,LLM_Input,LLM_output
20,claude-2.1,"[""Write a python code that calculates sum of 5...","[""Here is the python code that calculates the ..."
38,gpt-4-0613,"[""Hi who are you""]","[""Hello! I'm an AI developed to assist with ta..."
45,gpt-4-0613,"[""write the lyrics to a rap song about some du...","[""(Verse 1)\nYo, let's take a minute, prepare ..."
47,claude-2.1,"[""#include <stdio.h>\n#include <stdlib.h>\n#in...","[""This C program analyzes the contents of a te..."
48,gpt-4-0613,"[""Write OCL statement each Employee working in...","[""context Employee inv:\nself.age > 22""]"


####Create testing/training split using StratifiedKFold to guarantee class proportions with 80/20 split

In [6]:
X = DF[['LLM_Input', 'LLM_output']]
Y = DF['LLM_name']

dataSplit = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for fold, (train_idx, test_idx) in enumerate(dataSplit.split(X, Y), start=1):
    X_train = X.iloc[train_idx]
    X_test = X.iloc[test_idx]

    Y_train = Y.iloc[train_idx]
    Y_test = Y.iloc[test_idx]

    print(f"Fold {fold}: {len(X_train)} train, {len(X_test)} test")
    

X_train.to_csv('../data/processed/XTrain.csv', index=False)
X_test.to_csv('../data/processed/XTest.csv', index=False)
Y_train.to_csv('../data/processed/YTrain.csv', index=False)
Y_test.to_csv('../data/processed/YTest.csv', index=False)

Fold 1: 12140 train, 3036 test
Fold 2: 12141 train, 3035 test
Fold 3: 12141 train, 3035 test
Fold 4: 12141 train, 3035 test
Fold 5: 12141 train, 3035 test


##CNN MODEL

In [ ]:
import numpy as np
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix

#### Read in data and OHE for target column

In [ ]:
tf.keras.utils.set_random_seed(42)

X_train = pd.read_csv('../data/processed/XTrain.csv')
X_test = pd.read_csv('../data/processed/XTest.csv')
Y_train = pd.read_csv('../data/processed/YTrain.csv')
Y_test = pd.read_csv('../data/processed/YTest.csv')

X_train_Input = X_train['LLM_Input'].fillna("").astype(str).to_numpy()
X_train_Output = X_train['LLM_output'].fillna("").astype(str).to_numpy()

X_test_Input = X_test['LLM_Input'].fillna("").astype(str).to_numpy()
X_test_Output = X_test['LLM_output'].fillna("").astype(str).to_numpy()

encoder = LabelEncoder()
Y_train = encoder.fit_transform(Y_train)
Y_test = encoder.fit_transform(Y_test)

X_train_Output, val_train_Output, Y_train, Y_val = train_test_split(
    X_train_Output,
    Y_train,
    test_size=0.2,
    random_state=42,
    stratify=Y_train,
)


/opt/anaconda3/lib/python3.12/site-packages/sklearn/preprocessing/_label.py:114: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/opt/anaconda3/lib/python3.12/site-packages/sklearn/preprocessing/_label.py:114: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


####Tokenizing LLM responses

In [9]:
MAX_TOKENS = 20000
SEQUENCE_LENGTH = 1000

vectorizer = tf.keras.layers.TextVectorization(
    max_tokens=MAX_TOKENS,
    standardize="lower_and_strip_punctuation",
    split="whitespace",
    output_mode="int",
    output_sequence_length=SEQUENCE_LENGTH,
)

# Fit the vocabulary ONLY on the training subset.
vectorizer.adapt(
    tf.data.Dataset.from_tensor_slices(X_train_Output).batch(64)
)

lengths = pd.Series(X_train_Output).str.split().str.len()
print(lengths.describe(percentiles=[0.5, 0.9, 0.95, 0.99]))

count    12141.000000
mean       203.641463
std        214.237113
min          1.000000
50%        168.000000
90%        395.000000
95%        524.000000
99%       1016.600000
max       3876.000000
dtype: float64


2026-10-06 17:30:30.785130: I tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


In [10]:
inputs = tf.keras.Input(shape=(1,), dtype=tf.string, name="text")

x = vectorizer(inputs)

x = tf.keras.layers.Embedding(
    input_dim=len(vectorizer.get_vocabulary()),
    output_dim=128,
)(x)

x = tf.keras.layers.Conv1D(
    filters=128,
    kernel_size=5,
    activation="relu",
    padding="valid",
)(x)

x = tf.keras.layers.GlobalMaxPooling1D()(x)
x = tf.keras.layers.Dropout(0.5)(x)
x = tf.keras.layers.Dense(64, activation="relu")(x)

outputs = tf.keras.layers.Dense(
    3,
    activation="softmax",
)(x)

model = tf.keras.Model(inputs, outputs)

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ text (InputLayer)               │ (None, 1)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ text_vectorization              │ (None, 1000)           │             0 │
│ (TextVectorization)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding (Embedding)           │ (None, 1000, 128)      │     2,560,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d (Conv1D)                 │ (None, 996, 128)       │        82,048 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling1d            │ (None, 128)            │             0 │
│ (GlobalMaxPooling1D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 3)              │           195 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,650,499 (10.11 MB)

 Trainable params: 2,650,499 (10.11 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
def make_dataset(texts, labels, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices(
        (texts.reshape(-1, 1), labels)
    )
    if shuffle:
        ds = ds.shuffle(
            buffer_size=len(texts),
            seed=42,
            reshuffle_each_iteration=True,
        )
    return ds.batch(32).prefetch(tf.data.AUTOTUNE)

train_ds = make_dataset(X_train_Output, Y_train, shuffle=True)
val_ds = make_dataset(val_train_Output, Y_val)
test_ds = make_dataset(X_test_Output, Y_test)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=3,
            restore_best_weights=True,
        )
    ],
)

Epoch 1/20
380/380 ━━━━━━━━━━━━━━━━━━━━ 27s 70ms/step - accuracy: 0.5380 - loss: 0.9214 - val_accuracy: 0.7799 - val_loss: 0.5196
Epoch 2/20
380/380 ━━━━━━━━━━━━━━━━━━━━ 30s 79ms/step - accuracy: 0.8172 - loss: 0.4417 - val_accuracy: 0.8105 - val_loss: 0.4543
Epoch 3/20
380/380 ━━━━━━━━━━━━━━━━━━━━ 32s 85ms/step - accuracy: 0.9101 - loss: 0.2362 - val_accuracy: 0.8287 - val_loss: 0.4282
Epoch 4/20
380/380 ━━━━━━━━━━━━━━━━━━━━ 31s 81ms/step - accuracy: 0.9563 - loss: 0.1276 - val_accuracy: 0.8323 - val_loss: 0.5063
Epoch 5/20
380/380 ━━━━━━━━━━━━━━━━━━━━ 31s 82ms/step - accuracy: 0.9742 - loss: 0.0752 - val_accuracy: 0.8267 - val_loss: 0.5390
Epoch 6/20
380/380 ━━━━━━━━━━━━━━━━━━━━ 31s 81ms/step - accuracy: 0.9805 - loss: 0.0576 - val_accuracy: 0.8264 - val_loss: 0.6052


In [ ]:
test_loss, test_accuracy = model.evaluate(test_ds)

probabilities = model.predict(test_ds)
predicted_ids = probabilities.argmax(axis=1)

print("Test accuracy:", test_accuracy)

print(classification_report(
    Y_test,
    predicted_ids,
    labels=np.arange(3),
    target_names=[str(label) for label in encoder.classes_],
    zero_division=0,
))

print(confusion_matrix(
    Y_test,
    predicted_ids,
    labels=np.arange(3),
))

95/95 ━━━━━━━━━━━━━━━━━━━━ 2s 19ms/step - accuracy: 0.8329 - loss: 0.4213
95/95 ━━━━━━━━━━━━━━━━━━━━ 2s 18ms/step
Test accuracy: 0.828665554523468
                  precision    recall  f1-score   support

      claude-2.1       0.88      0.90      0.89      1116
      gpt-4-0613       0.79      0.85      0.82      1233
llama-2-70b-chat       0.82      0.67      0.74       686

        accuracy                           0.83      3035
       macro avg       0.83      0.81      0.82      3035
    weighted avg       0.83      0.83      0.83      3035

[[1004   95   17]
 [  97 1052   84]
 [  41  186  459]]
